# Repartition vs Coalesce in PySpark

This notebook presents hands-on examples and explanations for `repartition()` and `coalesce()` methods in PySpark DataFrames, highlighting their behavior, performance impact, and use cases. Includes comparison demos with row and partition counts, practical scenarios, and best practices for optimizing shuffle and minimizing unnecessary partitioning in Spark workloads.

## Why manage partitions? — Spark shuffle

- Efficient partitioning impacts performance in Spark jobs (especially large datasets)
- `repartition(n)` reshuffles to exactly n partitions (adds or reduces, always shuffles)
- `coalesce(n)` reduces the number of partitions without full shuffle; fast when shrinking partitions
- Over-partitioning slows down tasks; under-partitioning can create skew or underutilize resources

### Key differences — repartition() vs coalesce()

| Feature                  | repartition(n)         | coalesce(n)              |
|-------------------------|------------------------|--------------------------|
| Shuffle                  | Always (expensive)     | Only if increasing n     |
| Partition count change   | Any (increase/decrease)| Only decrease (or same)  |
| Typical use cases        | For wide reshuffles, balanced distribution | Efficient shrinking after a wide shuffle |
| Null/empty partition risk| Balanced after shuffle | May leave uneven partitions |

### Practical advice

- Use `repartition()` when you need a specific partition count or balanced data after a big shuffle (e.g., wide aggregations, joins).
- Use `coalesce()` after a shuffle if you need to reduce (and only reduce) the number of partitions efficiently — avoids full shuffle.
- Always check partition count before and after processing to avoid bottlenecks or resource waste:
  - Too many partitions can create overhead (especially small files on output)
  - Too few partitions concentrate workload, risking skew and slow performance
- For most output stages, aim for 1 partition per executor core for optimal parallelism.
- Avoid using coalesce() to increase partition count — it will trigger a shuffle; repartition() is better for that scenario.

---
Try these cells with larger datasets, inspect partition counts, and time operations for performance awareness.